# Blood Cell Detection with Deformable DETR — HS Mannheim

Fine-tuning **[SenseTime/deformable-detr](https://huggingface.co/SenseTime/deformable-detr)** (Hugging Face) on the BCCD dataset to detect three classes: **WBC**, **RBC**, and **Platelets**.

Deformable DETR uses multi-scale deformable attention, making it significantly better than standard DETR for small objects like Platelets.

| Detail | Value |
|---|---|
| Model | `SenseTime/deformable-detr` |
| Dataset | BCCD (VOC → COCO format) |
| Classes | WBC, RBC, Platelets |
| Metric | COCO mAP |
| Dependencies | `astral-uv` |
| Platform | Google Colab / Kaggle (GPU required) |

## 0. Environment Detection

Detects whether we are running on Google Colab or Kaggle and sets paths accordingly.

In [ ]:
import os, sys

ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_COLAB:
    WORK_DIR   = "/content/blood_cell_detection"
    SAVE_DIR   = "/content/drive/MyDrive/blood_cell_models"
    PLATFORM   = "Colab"
elif ON_KAGGLE:
    WORK_DIR   = "/kaggle/working/blood_cell_detection"
    SAVE_DIR   = "/kaggle/working/blood_cell_models"
    PLATFORM   = "Kaggle"
else:
    WORK_DIR   = os.getcwd()
    SAVE_DIR   = os.path.join(WORK_DIR, "saved_models")
    PLATFORM   = "Local"

print(f"Platform  : {PLATFORM}")
print(f"Work dir  : {WORK_DIR}")
print(f"Save dir  : {SAVE_DIR}")

## 1. Mount Google Drive *(Colab only)*

Keeps trained models and results permanently. Skip this cell on Kaggle.

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Drive mounted. Results will be saved to: {SAVE_DIR}")
else:
    print(f"Not on Colab — skipping Drive mount. Results saved to: {SAVE_DIR}")

## 2. Clone Repository

Pulls all scripts (`train.py`, `eval.py`, `tile.py`, etc.) from the private GitHub repository.

In [ ]:
import getpass

token = getpass.getpass("Enter your GitHub personal access token: ")

os.makedirs(os.path.dirname(WORK_DIR), exist_ok=True)
!rm -rf {WORK_DIR}
!git clone https://{token}@github.com/sahadvillan/blood_cell_detection.git {WORK_DIR}

%cd {WORK_DIR}
!ls -l

## 3. Install Dependencies

Installs all packages using `astral-uv`. On Kaggle, most packages are pre-installed so we use `uv pip install --system` directly.

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"

# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    print(f" GPU detected: {result.stdout.strip()}")
else:
    print("  No GPU detected — training will be very slow. Enable GPU in Runtime settings.")

# Install all dependencies from pyproject.toml lockfile
!uv sync
print("\n Dependencies installed.")

## 4. Prepare Dataset

Downloads the BCCD dataset from the official GitHub repository and converts annotations from VOC (XML) to COCO (JSON) format with train / val / test splits.

In [ ]:
!uv run python preprocess.py \
    https://github.com/Shenggan/BCCD_Dataset.git \
    ./blood_cell_data

# Verify splits
import json
for split in ["train", "val", "test"]:
    with open(f"blood_cell_data/{split}.json") as f:
        d = json.load(f)
    print(f"  {split:5s}: {len(d['images']):3d} images, {len(d['annotations']):4d} annotations")

## 5. Generate Tiled Training Set

Splits each 640×480 training image into four overlapping 380×300 tiles.
This doubles the effective size of Platelets in the feature maps, directly improving small-object detection.

- **Run once** — tiles are saved to `blood_cell_data/train_tiled/`
- Skip this cell if you want to train on full-resolution images (remove `--tiled` from the training cell)

In [ ]:
!uv run python tile.py

# Verify tiled dataset
with open("blood_cell_data/train_tiled.json") as f:
    tiled = json.load(f)
with open("blood_cell_data/train.json") as f:
    original = json.load(f)

from collections import Counter
cats = {c["id"]: c["name"] for c in original["categories"]}
orig_counts  = Counter(a["category_id"] for a in original["annotations"])
tiled_counts = Counter(a["category_id"] for a in tiled["annotations"])

print(f"\nImages : {len(original['images'])} → {len(tiled['images'])} tiles")
print(f"\n{'Class':12s}  {'Original':>10s}  {'Tiled':>10s}  {'Multiplier':>10s}")
print("-" * 48)
for cid, name in cats.items():
    mult = tiled_counts[cid] / orig_counts[cid]
    print(f"{name:12s}  {orig_counts[cid]:>10d}  {tiled_counts[cid]:>10d}  {mult:>9.1f}×")

## 6. Train Deformable DETR

Fine-tunes `SenseTime/deformable-detr` for 50 epochs with:

- **Image tiling** (`--tiled`): trains on 380×300 tiles instead of full 640×480 images, doubling platelet visibility
- **Platelet oversampling**: images containing platelets sampled 4× more per epoch
- **eos_coef = 0.04**: makes the model more sensitive to small objects
- **Cosine LR schedule** with 5% warmup: smooth convergence over 50 epochs

The best checkpoint (lowest validation loss) is automatically kept via `load_best_model_at_end=True`.

In [ ]:
!uv run python train.py --tiled

## 7. Evaluate Model

Runs the full evaluation pipeline on the **validation set**:
1. **COCO mAP** (standard metrics: mAP@50:95, mAP@50, per-size AP)
2. **Precision-Recall curve** per class at IoU=0.50
3. **Confusion matrix** (3 classes + background)
4. **10 validation image visualisations** (green = ground truth, red = predictions)

The `--tiled` flag runs inference on 2×2 tiles and merges predictions with NMS, matching the training approach.

In [ ]:
!uv run python eval.py --tiled

# Display key outputs inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, fname, title in zip(
    axes,
    ["eval_results/pr_curve.png", "eval_results/confusion_matrix.png"],
    ["Precision-Recall Curve (IoU=0.50)", "Confusion Matrix"]
):
    img = mpimg.imread(fname)
    ax.imshow(img)
    ax.set_title(title, fontsize=14)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Show Validation Visualisations

Displays the 10 validation images with ground truth boxes (green) and model predictions (red).

In [ ]:
import glob

vis_files = sorted(glob.glob("eval_results/val_visualizations/*.jpg"))
print(f"Found {len(vis_files)} validation visualisations\n")

cols = 2
rows = (len(vis_files) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 5))
axes = axes.flatten()

for ax, fpath in zip(axes, vis_files):
    img = mpimg.imread(fpath)
    ax.imshow(img)
    ax.set_title(os.path.basename(fpath).replace("vis_", ""), fontsize=9)
    ax.axis("off")

# Hide any unused subplots
for ax in axes[len(vis_files):]:
    ax.set_visible(False)

plt.suptitle("Validation Predictions  |  Green = Ground Truth  |  Red = Predicted", fontsize=13)
plt.tight_layout()
plt.show()